# Mixture of Adapters (MoA) — routed multi-expert LoRA building block (`core/adapters/moa`)

In [ ]:
import torch
import torch.nn as nn
from aligntune.core.adapters import MoALoraLayer

torch.manual_seed(0)

# Wrap a plain linear layer with N LoRA experts + a learned top-k router.
# Each token is routed to its top-2 most relevant experts, so capacity grows
# with num_experts while compute only grows with top_k.
base_linear = nn.Linear(64, 64)
moa_layer = MoALoraLayer(
    base_module=base_linear,
    num_experts=4,
    lora_r=8,
    lora_alpha=16,
    top_k=2,
    router_temp=1.0,
)

x = torch.randn(2, 10, 64)  # (batch, seq_len, hidden_dim)
out = moa_layer(x)
print("Output shape:", out.shape)
print("Router load-balance loss:", moa_layer.get_load_balance_loss())

In [ ]:
# The Mixture-of-Adapters recipe wires the same layer into a full SFT run
# with 4 experts / top-2 gating and a load-balance auxiliary loss, driven via
# the "moa" lora_variant + moa_* config knobs (see the shipped recipe):
#   recipes/configs/sft/llama3_moa_4experts.yaml
# and its Evolution-Strategies counterpart for router-only tuning:
#   recipes/configs/es/moa_router_tune.yaml